tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")


In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks "What\'s the weather like in Boston?" We need to get weather. Use function get_weather.', 'tool_calls': [{'id': 'fc_a839adcd-0a46-41b9-8a0e-d42dc8448c77', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 127, 'total_tokens': 176, 'completion_time': 0.104005489, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.025060197, 'prompt_tokens_details': None, 'queue_time': 0.315807689, 'total_time': 0.129065686}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0708ac49a5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a02b83-b7c8-7801-b8c6-9db7ffb085fb-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_a839adcd-0a46-41b9-8a0e-d42dc8448c77', 'type': 'tool_call'}] invalid_too

Manual Tool Calling
        ->
invoke model
        ->
check tool_calls
        ->
execute tool
        ->
send result to model
        ->
final answer

In [ ]:
#internally how tool calling happens
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in Boston right now.


In [14]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "What\'s the weather in Boston?" We need to use function get_weather with location "Boston". Use function call.', 'tool_calls': [{'id': 'fc_327258ef-d7f6-4ab5-9c2c-987dae313bcd', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 126, 'total_tokens': 180, 'completion_time': 0.11374127, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.004757997, 'prompt_tokens_details': None, 'queue_time': 0.369854704, 'total_time': 0.118499267}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a02b91-4d78-7b13-9e90-82384e2eb3ca-0', tool_calls=[{'name': 'get_weather', 'args': {